<img src="https://raw.githubusercontent.com/AnalyticJeremy/Olympics_Analysis/main/img/logos/olympics.svg" width="550" />

# Validate Data

Now that we have a bunch of data, we want to do a few quick sanity checks to make sure we got everything we were expecting.  We can do this by comparing the summary statistics that we scraped
with the details that we retrieved.  The details should add up to match the summary data.

In [0]:
library(SparkR)


Attaching package: ‘SparkR’

The following object is masked _by_ ‘.GlobalEnv’:

    setLocalProperty

The following objects are masked from ‘package:stats’:

    cov, filter, lag, na.omit, predict, sd, var, window

The following objects are masked from ‘package:base’:

    as.data.frame, colnames, colnames<-, drop, endsWith, intersect,
    rank, rbind, sample, startsWith, subset, summary, transform, union


In [0]:
countries <- read.df(path = "/olympics/raw/countries", source = "delta")
disciplines <- read.df(path = "/olympics/raw/disciplines", source = "delta")
edition_discipline_medal_summary <- read.df(path = "/olympics/raw/edition_discipline_medal_summary", source = "delta")
edition_disciplines <- read.df(path = "/olympics/raw/edition_disciplines", source = "delta")
edition_event_medals <- read.df(path = "/olympics/raw/edition_event_medals", source = "delta")
edition_events <- read.df(path = "/olympics/raw/edition_events", source = "delta")
edition_medal_summary <- read.df(path = "/olympics/raw/edition_medal_summary", source = "delta")
editions <- read.df(path = "/olympics/raw/editions", source = "delta")
display(editions)

#,Year,City,Country,Opened,Closed,Competition,Note,Url,Season,EditionID,ParticipantNote,MedalEventsNote,ParticipantCount,CountryCount,MedalEventCount,DisciplineCount
I,1896,Athina,https://olympedia-flags.s3.eu-central-1.amazonaws.com/GRE.png,6 April,15 April,6 – 13 April,,/editions/1,Summer,1.0,176 from 13 countries,43 in 10 disciplines,176.0,13.0,43.0,10.0
II,1900,Paris,https://olympedia-flags.s3.eu-central-1.amazonaws.com/FRA.png,,,14 May – 28 October,,/editions/2,Summer,2.0,1239 from 27 countries,95 in 22 disciplines,1239.0,27.0,95.0,22.0
III,1904,St. Louis,https://olympedia-flags.s3.eu-central-1.amazonaws.com/USA.png,14 May,,1 July – 26 November,,/editions/3,Summer,3.0,650 from 10 countries,95 in 18 disciplines,650.0,10.0,95.0,18.0
IV,1908,London,https://olympedia-flags.s3.eu-central-1.amazonaws.com/GBR.png,13 July,25 July,27 April – 31 October,,/editions/5,Summer,5.0,2025 from 23 countries,110 in 24 disciplines,2025.0,23.0,110.0,24.0
V,1912,Stockholm,https://olympedia-flags.s3.eu-central-1.amazonaws.com/SWE.png,6 July,15 July,5 May – 27 July,,/editions/6,Summer,6.0,2409 from 29 countries,107 in 19 disciplines,2409.0,29.0,107.0,19.0
VII,1920,Antwerpen,https://olympedia-flags.s3.eu-central-1.amazonaws.com/BEL.png,14 August,30 August,23 April – 12 September,,/editions/7,Summer,7.0,2680 from 29 countries,162 in 29 disciplines,2680.0,29.0,162.0,29.0
VIII,1924,Paris,https://olympedia-flags.s3.eu-central-1.amazonaws.com/FRA.png,5 July,27 July,4 May – 27 July,,/editions/8,Summer,8.0,3257 from 45 countries,131 in 23 disciplines,3257.0,45.0,131.0,23.0
IX,1928,Amsterdam,https://olympedia-flags.s3.eu-central-1.amazonaws.com/NED.png,28 July,12 August,17 May – 12 August,,/editions/9,Summer,9.0,3296 from 46 countries,125 in 20 disciplines,3296.0,46.0,125.0,20.0
X,1932,Los Angeles,https://olympedia-flags.s3.eu-central-1.amazonaws.com/USA.png,30 July,14 August,30 July – 14 August,,/editions/10,Summer,10.0,1924 from 47 countries,131 in 21 disciplines,1924.0,47.0,131.0,21.0
XI,1936,Berlin,https://olympedia-flags.s3.eu-central-1.amazonaws.com/GER.png,1 August,16 August,1 – 16 August,,/editions/11,Summer,11.0,4483 from 49 countries,149 in 28 disciplines,4483.0,49.0,149.0,28.0


In [0]:
# Do we have the right number of disciplines at each edition of the games?
ed_sum_df <- edition_disciplines |> group_by(edition_disciplines$EditionID) |> count();
joined_df <- join(alias(ed_sum_df, "esd"), alias(editions, "e"), ed_sum_df$EditionID == editions$EditionID, joinType="outer");
joined_df <- select(joined_df, "esd.EditionID", "count", "e.EditionID", "DisciplineCount");
display(joined_df |> filter(joined_df$count != joined_df$DisciplineCount))

EditionID,count,EditionID,DisciplineCount


In [0]:
# Do we have the right number of events at each edition of the games?
ee_sum_df <- edition_events |> group_by(edition_events$EditionID) |> count();
joined_df <- join(alias(ee_sum_df, "esd"), alias(editions, "e"), ee_sum_df$EditionID == editions$EditionID, joinType="outer");
joined_df <- select(joined_df, "esd.EditionID", "count", "e.EditionID", "MedalEventCount");
display(joined_df |> filter(joined_df$count != joined_df$MedalEventCount))

EditionID,count,EditionID,MedalEventCount


In [0]:
# Summarize the detailed medal data by country and edition.  Then compare it to the existing summary table and make sure everything matches up.
sdf <- edition_event_medals |>
          group_by("Country", "EditionID", "Medal") |>
          count() |>
          group_by("Country", "EditionID") |>
          pivot("Medal", c("Gold", "Silver", "Bronze")) |>
          sum("count") |>
          fillna(0)

sdf <- withColumn(sdf, "Total_Calc", sdf$Gold + sdf$Silver + sdf$Bronze);
sdf <- rename(sdf, Gold_Calc = sdf$Gold, Silver_Calc = sdf$Silver, Bronze_Calc = sdf$Bronze);
jdf <- join(sdf, alias(edition_medal_summary, "ems"), sdf$EditionID == edition_medal_summary$EditionID & sdf$Country == edition_medal_summary$NOC, joinType="outer") |> fillna(0)

display(jdf |> filter(jdf$Gold_Calc != jdf$Gold | jdf$Silver_Calc != jdf$Silver | jdf$Bronze_Calc != jdf$Bronze | jdf$Total_Calc != jdf$Total))

Country,EditionID,Gold_Calc,Silver_Calc,Bronze_Calc,Total_Calc,NOC,Gold,Silver,Bronze,Total,EditionID
null,1.0,0,1,11,12,null,0,0,0,0,0.0
null,2.0,0,3,8,11,null,0,0,0,0,0.0
null,3.0,0,3,7,10,null,0,0,0,0,0.0
null,5.0,0,6,12,18,null,0,0,0,0,0.0
null,6.0,1,4,7,12,null,0,0,0,0,0.0
null,7.0,2,8,22,32,null,0,0,0,0,0.0
null,8.0,1,0,2,3,null,0,0,0,0,0.0
null,9.0,2,2,1,5,null,0,0,0,0,0.0
null,10.0,1,1,6,8,null,0,0,0,0,0.0
null,11.0,3,3,3,9,null,0,0,0,0,0.0
